# CHUYÊN ĐỀ 1: CÁC PHƯƠNG PHÁP RỜI RẠC (DISCRETE REPRESENTATIONS) VÀ CODEC LLMS TRONG S2ST

Notebook này tập trung vào 2 trường phái nền tảng trong Speech-to-Speech Translation (S2ST):
1. **Discrete Semantic Units (HuBERT + k-means)**: Trường phái Textless S2ST của Meta (2021-2022).
2. **Neural Audio Codec & RVQ Tokens trong LLM**: Trường phái đưa âm thanh vào chung không gian từ điển với văn bản (AudioPaLM, SpeechGPT, VioLA - 2023).

> 💡 **Khả năng tương thích**: Notebook này có thể chạy trực tiếp trong VS Code / Jupyter hoặc chuyển đổi sang **Marimo** (`marimo convert ... -o ...py`) để chạy trên MoLab.


## 1. Cheat Sheet: Kích thước Tensor và Các Dạng Dữ liệu trong S2ST
Trước khi đi vào thuật toán, hãy nắm vững các chiều không gian (dimensions) của các dạng biểu diễn âm thanh:


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị đang sử dụng: {device}")

# TỔNG QUAN TENSOR SHAPES:
# 1. Raw Waveform:         [Batch, T_samples]       (ví dụ: 1s @ 16kHz = 16,000 samples)
# 2. Log Mel-Spectrogram:  [Batch, n_mels, T_frames] (80 hoặc 100 mel channels, ~50 hoặc 100 fps)
# 3. Discrete Units:       [Batch, T_units]         (chuỗi số nguyên int in [0, K-1], K ~ 100 - 1000)
# 4. Neural Codec Tokens:  [Batch, N_layers, T]     (EnCodec/DAC: N_layers=4-8, mỗi frame có 4-8 mã)
# 5. Whisper Continuous:   [Batch, T_whisper, 768]  (Vector liên tục 50Hz, giàu ngữ nghĩa)


## 2. Trường phái 1: Discrete Semantic Units (HuBERT + k-means)

### Xuất xứ học thuật:
- **Paper gốc**:
  - *Lee et al. (Meta AI, 2021)* - *"Direct Speech-to-Speech Translation With Discrete Units"*.
  - *Popuri et al. (Meta AI, 2022)* - *"Enhanced Direct Speech-to-Speech Translation Using Self-supervised Pre-trained Representations and UnitY"*.
  - *Jia et al. (Google, 2022)* - *"Translatotron 2: High-quality direct speech-to-speech translation with voice preservation"*.
- **Các bài báo cùng tư duy trong thư mục của bạn**: `Translatotron 2.pdf`, `Speech-to-speech survey.pdf`.

### Tư duy kiến trúc:
- Lớp biểu diễn trung gian (layer 6 hoặc layer 9 của HuBERT 12-layer) đã loại bỏ hầu hết thông tin người nói và tạp âm, chỉ giữ lại ngữ âm (phonetic/semantic).
- Thuật toán **k-means** (thường fit trên 50-100h dữ liệu với $K \in [100, 1000]$ cụm) biến mỗi frame thời gian 20ms thành 1 số nguyên (Unit ID).
- Nhờ đó, bài toán S2ST được giải quyết giống hệt Machine Translation: `Source Speech -> Target Discrete Units`.
- Cuối cùng, một bộ **Unit HiFi-GAN Vocoder** tái tạo dạng sóng từ các unit này.


In [ ]:
# Code mô phỏng: HuBERT SSL Feature Extractor + k-means Quantizer
class HuBERTKMeansTokenizer(nn.Module):
    """
    Lấy cảm hứng từ Lee et al. (Meta 2021) và UnitY (Popuri et al. 2022).
    Chuyển đổi vector ngữ nghĩa liên tục từ HuBERT thành Discrete Units.
    """
    def __init__(self, feat_dim=768, num_clusters=100):
        super().__init__()
        self.feat_dim = feat_dim
        self.num_clusters = num_clusters
        # Tâm cụm k-means (trong thực tế fit bằng Faiss)
        self.register_buffer("centroids", torch.randn(num_clusters, feat_dim))
        
    def forward(self, continuous_ssl_features):
        """
        Input:  [Batch, T_frames, feat_dim] (Vector liên tục từ HuBERT)
        Output: [Batch, T_frames] (Discrete Unit IDs in [0, num_clusters-1])
        """
        B, T, D = continuous_ssl_features.shape
        flat = continuous_ssl_features.reshape(-1, D)
        
        # Tính khoảng cách Euclidean tới các tâm cụm: ||x - c||^2
        dist = torch.cdist(flat, self.centroids) # [B*T, num_clusters]
        unit_ids = torch.argmin(dist, dim=-1)    # [B*T]
        return unit_ids.view(B, T)
        
    @staticmethod
    def deduplicate(units):
        """
        Loại bỏ các unit lặp lại liên tiếp (Run-Length Deduplication).
        Ví dụ: [12, 12, 12, 45, 45, 12] -> [12, 45, 12].
        Giúp chuỗi ngắn đi 2-3 lần, giảm tải tính toán cho Transformer.
        """
        res = []
        prev = None
        for u in units:
            if u != prev:
                res.append(u)
                prev = u
        return res

# Chạy thử nghiệm
hubert_tokenizer = HuBERTKMeansTokenizer(feat_dim=768, num_clusters=100).to(device)
fake_hubert_out = torch.randn(2, 50, 768, device=device) # 2 mẫu, mỗi mẫu 50 frames (1 giây)
discrete_units = hubert_tokenizer(fake_hubert_out)

print("Kích thước HuBERT feature liên tục: ", fake_hubert_out.shape)
print("Kích thước Discrete Units:         ", discrete_units.shape)
print("Chuỗi Units mẫu 0:                 ", discrete_units[0, :12].tolist())
print("Chuỗi Units sau Deduplication:     ", HuBERTKMeansTokenizer.deduplicate(discrete_units[0, :12].tolist()))


## 3. Trường phái 2: Neural Codec RVQ Tokens trong LLM

### Xuất xứ học thuật:
- **Paper gốc**:
  - *AudioPaLM (Rubenstein et al., Google 2023)* - *"AudioPaLM: A Large Language Model That Can Speak and Listen"*.
  - *SpeechGPT (Zhang et al., 2023)* - *"SpeechGPT: Empowering Large Language Models with Intrinsic Cross-Modal Conversational Abilities"*.
  - *VioLA (Wang et al., Microsoft 2023)* - *"VioLA: Unified Codec Language Models for Speech Recognition, Synthesis, and Translation"*.
  - *EnCodec (Défossez et al., Meta 2022)* - *"High Fidelity Neural Audio Compression"*.
- **Các bài báo cùng tư duy trong thư mục của bạn**: `speechgpt.pdf`, `AudioPaLM.pdf`, `viola.pdf`.

### Tư duy kiến trúc:
- **Hạn chế của HuBERT k-means**: Nó chỉ chứa ngữ nghĩa (semantics), mất sạch âm sắc người nói (timbre) và ngữ điệu tự nhiên.
- **Giải pháp RVQ (Residual Vector Quantization)**:
  - Chia âm thanh thành nhiều tầng (ví dụ 4 đến 8 codebooks).
  - Tầng 1: Đại diện cho cấu trúc thô và âm vị.
  - Các tầng sau: Bù đắp phần dư (residuals) chứa chi tiết âm sắc, hơi thở, cao độ.
- **Đưa vào LLM**: Mở rộng từ điển của LLM: Gom Text Tokens + Semantic Tokens + Acoustic Tokens. LLM học dự đoán token tiếp theo theo kiểu tự hồi quy (Autoregressive).


In [ ]:
# Code mô phỏng: Residual Vector Quantizer (RVQ) 4 tầng & Interleaved Tokenization
class ToyRVQCodec(nn.Module):
    """
    Lấy cảm hứng từ EnCodec (Défossez et al. 2022) và AudioPaLM (Google 2023).
    """
    def __init__(self, num_codebooks=4, codebook_size=1024, latent_dim=128):
        super().__init__()
        self.num_codebooks = num_codebooks
        self.codebooks = nn.ModuleList([
            nn.Embedding(codebook_size, latent_dim) for _ in range(num_codebooks)
        ])
        
    def encode(self, continuous_latents):
        """
        Input:  [Batch, T_frames, latent_dim]
        Output: [Batch, num_codebooks, T_frames] (Chuỗi mã rời rạc ở từng tầng)
        """
        residual = continuous_latents
        codes = []
        for cb in self.codebooks:
            # Tìm code vector gần nhất trong codebook hiện tại
            dist = torch.cdist(residual, cb.weight.unsqueeze(0)) # [Batch, T, codebook_size]
            indices = torch.argmin(dist, dim=-1)                # [Batch, T]
            codes.append(indices)
            
            # Tính phần dư cho tầng tiếp theo: residual = residual - quantized
            quantized = cb(indices)
            residual = residual - quantized
        return torch.stack(codes, dim=1)

# Chạy thử nghiệm
rvq = ToyRVQCodec(num_codebooks=4, codebook_size=1024, latent_dim=128).to(device)
fake_audio = torch.randn(2, 25, 128, device=device) # 25 frames
rvq_tokens = rvq.encode(fake_audio)

print("Kích thước Audio Latents đầu vào: ", fake_audio.shape)
print("Kích thước RVQ Tokens:            ", rvq_tokens.shape)
print("Tầng 1 (Coarse/Phonetic):         ", rvq_tokens[0, 0, :5].tolist())
print("Tầng 4 (Fine/Acoustic detail):    ", rvq_tokens[0, 3, :5].tolist())


## 4. Tóm kết Chuyên đề 1: Ưu điểm và Điểm nghẽn
- **Ưu điểm**:
  - Dễ dàng tích hợp vào hạ tầng Transformer và LLM có sẵn (chỉ cần dùng Cross-Entropy Loss chuẩn).
- **Điểm nghẽn (Bottleneck)**:
  - Chuỗi RVQ tokens cực dài (ví dụ 8 tầng $	imes$ 75 fps = **600 tokens/giây**).
  - Ép LLM gánh chuỗi dài như vậy làm bộ nhớ Self-Attention ($O(N^2)$) bùng nổ, dễ gây lặp từ (looping) hoặc nuốt chữ.
  - Đây chính là lý do thôi thúc các nhà nghiên cứu chuyển dịch sang **Biểu diễn Liên tục và Flow Matching**, được trình bày ở **Chuyên đề 2**!
